In [2]:
# Cell 1 - Imports
import requests
import pandas as pd
import time


In [3]:
# Cell 2 — Configuration (only this cell needs to be updated)

BASE_URL = "https://findata-prod.worldbank.org:9047/"   # PROD
TOKEN    = "tTQlvmAkRWKs8Kc5pOy0XFU+yvAPwsMlSwy9ZVduLyk6k6FhAhE22k4GT7wqHQ==" 
SCHEMA   = ["publishFINDATAREP", "IFCTRE", "Application", "Murex"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# ── Date parameter — update this to change the filter ──
BUSINESS_DATE = "2026-08-24"

# ── All your queries ──
QUERIES = {
    "perf_attribution"      : 'SELECT * FROM "Perf_Attribution"',
    "perf_benchmark"        : 'SELECT * FROM "Perf_Benchmark_Weights"',
    "perf_vs_funding"       : 'SELECT * FROM "Perf_Vs_Funding_Snap"',
    "plvar_snapshot"        : 'SELECT * FROM "Plvar_Snapshot"',
    "portfolio_details"     : 'SELECT * FROM "Portfolio_Details"',
    "securities_detail"     : 'SELECT * FROM "Securities_Detail"',
    "trade_mktval_detail"   : 'SELECT * FROM "Trade_Mktval_Detail"',
    "security_prices_detail": 'SELECT * FROM "Security_Prices_Detail"',
    "inv_pnl_summary"       : f'SELECT * FROM "Inv_Pnl_Summary" WHERE "Business_Date" = \'{BUSINESS_DATE}\'',
    "inv_security_position" : 'SELECT * FROM "Inv_Security_Position"',
}

# ── Only change this line to switch query ──
ACTIVE_QUERY =  "securities_detail"
SQL_QUERY    = QUERIES[ACTIVE_QUERY]


In [4]:
# ============================================================
# Cell 1: Imports
# ============================================================
import os, time, requests
import pandas as pd

# ============================================================
# Cell 2: Configuration (only this cell needs to be updated)
# ============================================================
# QA   : https://findata-qa.worldbank.org:9047
# DEV  : https://findata-dev.worldbank.org:9047
# PROD : https://findata-prod.worldbank.org:9047   <- confirm exact host with IT
BASE_URL = "https://findata-prod.worldbank.org:9047".rstrip("/")

# Token is specific to each environment. It is asked for at run time.
TOKEN    = "tTQlvmAkRWKs8Kc5pOy0XFU+yvAPwsMlSwy9ZVduLyk6k6FhAhE22k4GT7wqHQ==" 

# Confirm the schema for the target environment with IT
SCHEMA = ["publishFINDATAREP", "IFCTRE", "Operational_Data", "Common"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
SCHEMA_PATH = ".".join(SCHEMA)

# ============================================================
# Cell 3: Query function
# ============================================================
def run_query(sql, poll=1.0, page=500):
    """Run SQL on Dremio and return the result as a DataFrame."""
    r = requests.post(f"{BASE_URL}/api/v3/sql", headers=HEADERS, json={"sql": sql})
    r.raise_for_status()
    job_id = r.json()["id"]

    while True:
        s = requests.get(f"{BASE_URL}/api/v3/job/{job_id}", headers=HEADERS)
        s.raise_for_status()
        info = s.json()
        state = info["jobState"]
        if state == "COMPLETED":
            break
        if state in ("FAILED", "CANCELED"):
            raise RuntimeError(f"Job {state}: {info.get('errorMessage')}")
        time.sleep(poll)

    rows, offset = [], 0
    while True:
        res = requests.get(
            f"{BASE_URL}/api/v3/job/{job_id}/results",
            headers=HEADERS,
            params={"offset": offset, "limit": page},
        )
        res.raise_for_status()
        data = res.json()
        rows.extend(data["rows"])
        offset += page
        if offset >= data["rowCount"]:
            break
    return pd.DataFrame(rows)

# ============================================================
# Cell 4: Connection test
# ============================================================
print(run_query("SELECT 1 AS ok"))

# ============================================================
# Cell 5: Find the Murex columns for the PL_Summary extract
# ============================================================
sql = f"""
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA."COLUMNS"
WHERE TABLE_SCHEMA = '{SCHEMA_PATH}'
"""
cols = run_query(sql)

# Fallback if nothing comes back: schema naming or case may differ
if cols.empty:
    print(f"No columns found for '{SCHEMA_PATH}'. Schemas that look similar:")
    print(run_query("""
        SELECT DISTINCT TABLE_SCHEMA
        FROM INFORMATION_SCHEMA."COLUMNS"
        WHERE TABLE_SCHEMA LIKE '%IFCTRE%' OR TABLE_SCHEMA LIKE '%FINDATAREP%'
    """).to_string(index=False))
else:
    print(f"{cols['TABLE_NAME'].nunique()} tables, {len(cols)} columns found in {SCHEMA_PATH}")

    keywords = {
        "trade_date":    ["TRADE", "TRD", "DEAL", "EXEC"],
        "settle_date":   ["SETTL", "VALUE", "VAL_DATE", "DELIV"],
        "maturity_date": ["MATUR", "MAT_", "EXPIR", "END_DATE"],
        "notional":      ["NOTIONAL", "NOMINAL", "NOTION", "PRINCIPAL", "FACE", "AMOUNT", "QUANTITY"],
    }
    date_types = ("DATE", "TIMESTAMP")

    for field, kws in keywords.items():
        mask = cols["COLUMN_NAME"].str.upper().apply(lambda c: any(k in c for k in kws))
        hits = cols[mask]
        if field != "notional":
            hits = hits[hits["DATA_TYPE"].str.upper().str.contains("|".join(date_types))]
        print(f"\n=== Candidates for {field} ===")
        print(hits.sort_values(["TABLE_NAME", "COLUMN_NAME"]).to_string(index=False)
              if not hits.empty else "No match")


   ok
0   1
110 tables, 6063 columns found in publishFINDATAREP.IFCTRE.Operational_Data.Common

=== Candidates for trade_date ===
              TABLE_NAME       COLUMN_NAME DATA_TYPE
      Bond_League_Detail        Trade_date      DATE
     Bond_League_Summary        Trade_date      DATE
       Bond_Trade_Detail        Trade_Date      DATE
    Bond_Trade_Detail_ME        Trade_Date      DATE
       Broker_Fee_Detail        TRADE_DATE      DATE
         Cashflow_Detail        Trade_date      DATE
           Ffopfdndealse      MX_TRADE_DAT      DATE
           Ffopfdnnamese      MX_TRADE_DAT      DATE
           Ffopfdnsectse      MX_TRADE_DAT      DATE
           Ffopfdntypese      MX_TRADE_DAT      DATE
   Inv_Security_Position      MX_TRADE_DAT      DATE
Inv_Security_Position_ME      MX_TRADE_DAT      DATE
            Md_Audit_Tbl       MX_TRADE_DT      DATE
            Murex_Trades     MX_TRADE_DATE      DATE
  Trade_And_Asset_Detail        Trade_Date      DATE
  Trade_And_Asset_Deta

In [3]:
# Cell 3 — Display function
def display_df(df):
    return df.style.set_properties(**{
        'white-space': 'nowrap',
        'font-size': '11px',
        'text-align': 'left',
        'border': '1px solid lightgrey',
        'color': 'black'
    }).set_table_styles([
        {
            'selector': 'thead th',
            'props': [
                ('white-space', 'nowrap'),
                ('font-size', '11px'),
                ('font-weight', 'bold'),
                ('background-color', '#4472C4'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('border', '1px solid lightgrey')
            ]
        },
        {
            'selector': 'tbody tr:nth-child(even)',
            'props': [('background-color', '#f2f2f2'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:nth-child(odd)',
            'props': [('background-color', 'white'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:hover',
            'props': [('background-color', '#d6e4f0'), ('color', 'black')]
        }
    ])


In [4]:
# Cell 4 — Submit the SQL job
response = requests.post(
    f"{BASE_URL}/api/v3/sql",
    headers=HEADERS,
    json={
        "sql": SQL_QUERY,
        "context": SCHEMA
    },
    verify=True,
    timeout=30
)
response.raise_for_status()
job_id = response.json().get("id")
print(f"Job submitted successfully.")
print(f"Job ID: {job_id}")


Job submitted successfully.
Job ID: 154bd737-5198-f485-69e3-69d67bcc3200


In [5]:
# Cell 5 — Poll until job completes
url = f"{BASE_URL}/api/v3/job/{job_id}"
for attempt in range(30):
    response = requests.get(url, headers=HEADERS, verify=True)
    response.raise_for_status()
    status = response.json().get("jobState")
    print(f"Attempt {attempt + 1}: {status}")
    if status == "COMPLETED":
        print("Job completed!")
        break
    elif status in ("FAILED", "CANCELED"):
        print(f"Job failed: {response.json().get('errorMessage')}")
        break
    time.sleep(2)


Attempt 1: COMPLETED
Job completed!


In [6]:
# Cell 6 — Fetch results and build DataFrame

# ── Column selection per query ──
COLUMNS = {
    "perf_attribution"      : [],
    "perf_benchmark"        : [],
    "perf_vs_funding"       : [],
    "plvar_snapshot"        : [],
    "portfolio_details"     : [],
    "securities_detail"     : [],
    "trade_mktval_detail"   : [],
    "security_prices_detail": [],
    "inv_pnl_summary"       : [
        "Business_Date", "MX_L3_PORTFO", "MX_L4_PORTFO", "Currency", "MX_CD_ELIG", "Trade_Typology",
        "Instrument", "Contract_Origin", "Trade_ID",
        "Country", "CTP_Code", "MX_DEAL_TYPE", "DTD_pnl_usd", "MTD_Pnl_USD",
        "YTD_pnl_USD", "LTD_pnl_usd", "Cash", "Cash_USD", "Market_Value_USD"
    ],
    "inv_security_position" : [],
}

# ── Rename mapping per query ──
RENAME = {
    "inv_pnl_summary" : {
        "Business_Date"   : "Business_Date",
        "MX_L3_PORTFO"   : "L3_PORTFO",
        "MX_L4_PORTFO"   : "L4_PORTFO",
        "Currency"        : "CURRENCY",
        "MX_CD_ELIG"     : "CD_ELIG",
        "Trade_Typology"  : "TYPOLOGY",
        "Instrument"      : "INSTRUMEN",
        "Contract_Origin" : "CNT_ORG",
        "Trade_ID"        : "NB",
        "Country"         : "COUNTRY",
        "CTP_Code"        : "CTP",
        "MX_DEAL_TYPE"   : "DEAL_TYPE",
        "DTD_pnl_usd"    : "DTD_PL_US",
        "MTD_Pnl_USD"    : "MTD_PL_US",
        "YTD_pnl_USD"    : "YTD_PL_US",
        "LTD_pnl_usd"    : "LTD_PL_US",
        "Cash"            : "CASH",
        "Cash_USD"        : "CASH_US",
        "Market_Value_USD": "MV_US",
    },
}

# ── Fetch first page ──
response = requests.get(
    f"{BASE_URL}/api/v3/job/{job_id}/results?offset=0&limit=500",
    headers=HEADERS,
    verify=True
)
response.raise_for_status()
data = response.json()

columns  = [col["name"] for col in data.get("schema", [])]
all_rows = data.get("rows", [])

# ── Paginate if more than 500 rows ──
offset = 500
while True:
    r = requests.get(
        f"{BASE_URL}/api/v3/job/{job_id}/results?offset={offset}&limit=500",
        headers=HEADERS,
        verify=True
    )
    batch = r.json().get("rows", [])
    if not batch:
        break
    all_rows.extend(batch)
    offset += len(batch)
    print(f"Fetched {offset} rows so far...")

df = pd.DataFrame(all_rows, columns=columns)

# ── Inspect all available columns before filtering ──
print(f"\nAll available columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

# ── Apply column selection ──
selected_cols = COLUMNS.get(ACTIVE_QUERY, [])
if selected_cols:
    missing = [c for c in selected_cols if c not in df.columns]
    if missing:
        print(f"\n⚠️  Columns not found and skipped: {missing}")
    df = df[[c for c in selected_cols if c in df.columns]]

# ── Apply column renaming ──
rename_map = RENAME.get(ACTIVE_QUERY, {})
if rename_map:
    df = df.rename(columns=rename_map)

print(f"\nActive Query  : {ACTIVE_QUERY}")
print(f"Total Rows    : {len(df)}")
print(f"Total Columns : {len(df.columns)}")
print(f"Columns       : {list(df.columns)}")


Fetched 1000 rows so far...
Fetched 1500 rows so far...
Fetched 2000 rows so far...
Fetched 2500 rows so far...
Fetched 3000 rows so far...
Fetched 3500 rows so far...
Fetched 4000 rows so far...
Fetched 4500 rows so far...
Fetched 5000 rows so far...
Fetched 5500 rows so far...
Fetched 6000 rows so far...
Fetched 6500 rows so far...
Fetched 7000 rows so far...
Fetched 7500 rows so far...
Fetched 8000 rows so far...
Fetched 8500 rows so far...
Fetched 9000 rows so far...
Fetched 9500 rows so far...
Fetched 10000 rows so far...
Fetched 10500 rows so far...
Fetched 11000 rows so far...
Fetched 11500 rows so far...
Fetched 12000 rows so far...
Fetched 12500 rows so far...
Fetched 13000 rows so far...
Fetched 13500 rows so far...
Fetched 14000 rows so far...
Fetched 14500 rows so far...
Fetched 15000 rows so far...
Fetched 15500 rows so far...
Fetched 16000 rows so far...
Fetched 16500 rows so far...
Fetched 17000 rows so far...
Fetched 17500 rows so far...
Fetched 18000 rows so far...
Fet

In [2]:
mask = (
    df["L3_PORTFO"].isin(["LAM", "LAM_LIQUIDTY", "LAM_STRATEGIC"]) &
    (df["TYPOLOGY"] == "Investment Bond")
)

df_filtered = df[mask].reset_index(drop=True)

print(f"Rows before filter : {len(df)}")
print(f"Rows after filter  : {len(df_filtered)}")


NameError: name 'df' is not defined

In [1]:
display_df(df.head(10))   # first 10 rows

NameError: name 'display_df' is not defined

In [41]:
display_df(df_filtered.tail(10))   # last 10 rows

,Business_Date,L3_PORTFO,L4_PORTFO,CURRENCY,CD_ELIG,TYPOLOGY,INSTRUMEN,CNT_ORG,NB,COUNTRY,CTP,DEAL_TYPE,DTD_PL_US,MTD_PL_US,YTD_PL_US,LTD_PL_US,CASH,CASH_US,MV_US
721,2026-07-24,LAM_STRATEGIC,LAM_STG_GRE,CHF,N,Investment Bond,KBBS 0.15 020427 S,1700884,1701470,CH,BNPAFR,Asset,537.730000,78745.650000,78745.650000,78745.650000,-6029028.667480,-6029028.667480,6107774.319406
722,2026-07-24,LAM_STRATEGIC,LAM_STG_GRE,AUD,N,Investment Bond,KOMMUN 2.9 271126 S MTN,1701408,1701994,DK,TDOMGB,Asset,62.130000,50667.200000,50667.200000,50667.200000,-1252420.443793,-1252420.443793,1303087.639130
723,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NDAFH 1 011027,1702377,1702963,DK,ESSESW,Asset,21023.050000,4620205.020000,4620205.020000,4620205.020000,-40457512.992219,-40457512.992219,45077718.010013
724,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NDAFH 1 011027,1701561,1702147,DK,NDEAHE,Asset,-24311.480000,-240356.720000,-240356.720000,-240356.720000,44837361.294558,44837361.294558,-45077718.010013
725,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NDAFH 1 011027,1701918,1702504,DK,NDEAHE,Asset,-24313.180000,-238094.620000,-238094.620000,-238094.620000,44839623.393410,44839623.393410,-45077718.010013
726,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NDAFH 1 011027,1701378,1701964,DK,NYKBDK,Asset,21434.320000,4072438.140000,4072438.140000,4072438.140000,-41005279.866823,-41005279.866823,45077718.010013
727,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NYKRE 1 010127 13H*,1701772,1702358,DK,ESSESW,Asset,2904.340000,3038853.260000,3038853.260000,3038853.260000,-42506274.676347,-42506274.676347,45545127.931468
728,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NYKRE 1 010127 13H*,1701573,1702159,DK,JYBADK,Asset,3635.790000,-237746.410000,-237746.410000,-237746.410000,-30601165.028192,-30601165.028192,30363418.620979
729,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NYKRE 1 010127 13H*,1701598,1702184,DK,JYBADK,Asset,1630.820000,-139402.570000,-139402.570000,-139402.570000,-13542726.454774,-13542726.454774,13403323.881859
730,2026-07-24,LAM_STRATEGIC,LAM_STG_CB,DKK,N,Investment Bond,NYKRE 1 010127 13H*,1701741,1702327,DK,JYBADK,Asset,3847.040000,-249403.550000,-249403.550000,-249403.550000,-32391207.596000,-32391207.596000,32141804.049609


In [42]:
# Cell — Export DataFrame to Excel with professional formatting
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os
from datetime import datetime

# ── Build filename with date ──
date_str  = BUSINESS_DATE.replace("-", "")           # e.g. 20260811
filename  = f"MX_Subaccount_PL_{date_str}.xlsx"

output_dir = r"C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\4. Projects & Initiatives\Murex_Testing_LAM_Cash\08_DR\DR4\01.Inputs\1_Datamart_Extracts\4_Murex_PL_Summary"

os.makedirs(output_dir, exist_ok=True)
filepath  = os.path.join(output_dir, filename)

# ── Save raw DataFrame first ──
df_filtered.to_excel(filepath, index=False, sheet_name="PL_Summary")

# ── Apply professional formatting ──
wb = load_workbook(filepath)
ws = wb["PL_Summary"]

# Header style
header_font    = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
header_fill    = PatternFill(fill_type="solid", fgColor="003366")   # WBG dark blue
header_align   = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Body style
body_font      = Font(name="Calibri", size=10)
body_align_l   = Alignment(horizontal="left",   vertical="center")
body_align_r   = Alignment(horizontal="right",  vertical="center")

# Border
thin           = Side(style="thin", color="CCCCCC")
border         = Border(left=thin, right=thin, top=thin, bottom=thin)

# Numeric columns to format
numeric_cols   = ["DTD_PL_US", "MTD_PL_US", "YTD_PL_US", "LTD_PL_US", "CASH", "CASH_US", "MV_US"]
num_format     = '#,##0.00'

# ── Format header row ──
for cell in ws[1]:
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = header_align
    cell.border    = border

ws.row_dimensions[1].height = 30

# ── Format body rows ──
col_indices = {cell.value: cell.column for cell in ws[1]}

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.font   = body_font
        cell.border = border
        col_name    = ws.cell(row=1, column=cell.column).value
        if col_name in numeric_cols:
            cell.number_format = num_format
            cell.alignment     = body_align_r
        else:
            cell.alignment     = body_align_l

    # Alternating row color
    if row[0].row % 2 == 0:
        for cell in row:
            cell.fill = PatternFill(fill_type="solid", fgColor="EEF2F7")

# ── Auto-fit column widths ──
for col in ws.columns:
    max_len = max((len(str(cell.value)) if cell.value else 0) for cell in col)
    ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 40)

# ── Freeze header row ──
ws.freeze_panes = "A2"

# ── Add auto filter ──
ws.auto_filter.ref = ws.dimensions

wb.save(filepath)
print(f"✅ File saved : {filepath}")


✅ File saved : C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\4. Projects & Initiatives\Murex_Testing_LAM_Cash\08_DR\DR4\01.Inputs\1_Datamart_Extracts\4_Murex_PL_Summary\MX_Subaccount_PL_20260724.xlsx


In [ ]:
# Cell 7 — Display the DataFrame
#display_df(df.head(10))   # first 10 rows
#display_df(df.tail(10))   # last 10 rows
display_df(df_filtered)            # everything



In [ ]:
df.tail()

In [ ]:
# Cell 8 — Optional: Save to CSV or Excel
df.to_csv("output.csv", index=False)
print("Saved to output.csv")

df.to_excel("output.xlsx", index=False)
print("Saved to output.xlsx")


In [ ]:
print(df.columns.tolist())
